# TECHIN 516 Lab 6
### name: Anthony Chen

### Task 1: Code

```map annotator```

In [ ]:
import rclpy
from rclpy.node import Node
from geometry_msgs.msg import PoseStamped, PoseArray, Pose
from tf2_ros import Buffer, TransformListener
import pandas as pd
import os
import sys
from threading import Thread

class MapAnnotator(Node):
    def __init__(self):
        super().__init__('map_annotator')
        
        # Initialize TF buffer and listener to get robot's current pose
        self.tf_buffer = Buffer()
        self.tf_listener = TransformListener(self.tf_buffer, self)
        
        # Publisher for navigation goals
        self.goal_pub = self.create_publisher(
            PoseStamped, 
            '/goal_pose', 
            10
        )
        
        # Publisher for pose array (continuously publish saved poses)
        self.pose_array_pub = self.create_publisher(
            PoseArray,
            '/saved_poses',
            10
        )
        
        # Timer to continuously publish pose array
        self.create_timer(1.0, self.publish_pose_array)
        
        # Dictionary to store poses {name: PoseStamped}
        self.poses = {}
        
        # CSV file path
        self.csv_file = 'saved_poses.csv'
        
        # Load existing poses from CSV
        self.load_poses()
        
        # Print welcome message and commands
        self.print_commands()
        
    def print_commands(self):
        """Print available commands"""
        print("\ncommands:")
        print("- list")
        print("- save <name>")
        print("- delete <name>")
        print("- goto <name>")
        print("- exit")
        print("- help\n")
        
    def load_poses(self):
        """Load poses from CSV file if it exists"""
        if os.path.exists(self.csv_file):
            try:
                df = pd.read_csv(self.csv_file)
                for _, row in df.iterrows():
                    pose_stamped = PoseStamped()
                    pose_stamped.header.frame_id = 'map'
                    pose_stamped.pose.position.x = row['px']
                    pose_stamped.pose.position.y = row['py']
                    pose_stamped.pose.position.z = row['pz']
                    pose_stamped.pose.orientation.x = row['ox']
                    pose_stamped.pose.orientation.y = row['oy']
                    pose_stamped.pose.orientation.z = row['oz']
                    pose_stamped.pose.orientation.w = row['ow']
                    self.poses[row['name']] = pose_stamped
                self.get_logger().info(f'Loaded {len(self.poses)} poses from {self.csv_file}')
            except Exception as e:
                self.get_logger().error(f'Error loading poses: {str(e)}')
        
    def save_poses(self):
        """Save all poses to CSV file"""
        if not self.poses:
            # If no poses, create empty file or remove existing
            if os.path.exists(self.csv_file):
                os.remove(self.csv_file)
            return
            
        data = []
        for name, pose_stamped in self.poses.items():
            data.append({
                'name': name,
                'px': pose_stamped.pose.position.x,
                'py': pose_stamped.pose.position.y,
                'pz': pose_stamped.pose.position.z,
                'ox': pose_stamped.pose.orientation.x,
                'oy': pose_stamped.pose.orientation.y,
                'oz': pose_stamped.pose.orientation.z,
                'ow': pose_stamped.pose.orientation.w
            })
        
        df = pd.DataFrame(data)
        df.to_csv(self.csv_file, index=False)
        self.get_logger().info(f'Saved {len(self.poses)} poses to {self.csv_file}')
        
    def get_current_pose(self):
        """Get the current pose of the robot in the map frame"""
        try:
            # Get transform from map to base_link (robot's current position)
            transform = self.tf_buffer.lookup_transform(
                'map',
                'base_link',
                rclpy.time.Time(),
                timeout=rclpy.duration.Duration(seconds=1.0)
            )
            
            # Create PoseStamped from transform
            pose_stamped = PoseStamped()
            pose_stamped.header.frame_id = 'map'
            pose_stamped.header.stamp = self.get_clock().now().to_msg()
            pose_stamped.pose.position.x = transform.transform.translation.x
            pose_stamped.pose.position.y = transform.transform.translation.y
            pose_stamped.pose.position.z = transform.transform.translation.z
            pose_stamped.pose.orientation = transform.transform.rotation
            
            return pose_stamped
        except Exception as e:
            self.get_logger().error(f'Could not get current pose: {str(e)}')
            return None
            
    def publish_pose_array(self):
        """Continuously publish all saved poses as a PoseArray"""
        if not self.poses:
            return
            
        pose_array = PoseArray()
        pose_array.header.frame_id = 'map'
        pose_array.header.stamp = self.get_clock().now().to_msg()
        
        for pose_stamped in self.poses.values():
            pose_array.poses.append(pose_stamped.pose)
            
        self.pose_array_pub.publish(pose_array)
        
    def cmd_list(self):
        """List all saved poses"""
        if not self.poses:
            print("No poses")
        else:
            print("Poses:")
            for name in self.poses.keys():
                print(f"- {name}")
                
    def cmd_save(self, name):
        """Save current pose with given name"""
        if not name:
            print("Error: Please provide a name for the pose")
            return
            
        current_pose = self.get_current_pose()
        if current_pose is None:
            print("Error: Could not get current robot pose")
            return
            
        self.poses[name] = current_pose
        print(f"Saved pose '{name}'")
        
    def cmd_delete(self, name):
        """Delete pose by name"""
        if not name:
            print("Error: Please provide a name to delete")
            return
            
        if name in self.poses:
            del self.poses[name]
            print(f"Deleted pose '{name}'")
        else:
            print(f"'{name}' does not exist")
            
    def cmd_goto(self, name):
        """Send navigation goal to named pose"""
        if not name:
            print("Error: Please provide a name to go to")
            return
            
        if name not in self.poses:
            print(f"'{name}' does not exist")
            return
            
        # Publish the pose as a navigation goal
        pose_stamped = self.poses[name]
        pose_stamped.header.stamp = self.get_clock().now().to_msg()
        self.goal_pub.publish(pose_stamped)
        print(f"Navigating to '{name}'")
        
    def process_command(self, command):
        """Process user command"""
        command = command.strip()
        if not command:
            return True
            
        parts = command.split(maxsplit=1)
        cmd = parts[0].lower()
        arg = parts[1] if len(parts) > 1 else ""
        
        if cmd == 'list':
            self.cmd_list()
        elif cmd == 'save':
            self.cmd_save(arg)
        elif cmd == 'delete':
            self.cmd_delete(arg)
        elif cmd == 'goto':
            self.cmd_goto(arg)
        elif cmd == 'help':
            self.print_commands()
        elif cmd == 'exit':
            print("Saving poses and exiting...")
            self.save_poses()
            return False
        else:
            print(f"Unknown command: '{cmd}'. Type 'help' for available commands.")
            
        return True
        
    def run(self):
        """Main run loop for user input"""
        running = True
        while running and rclpy.ok():
            try:
                user_input = input("> ")
                running = self.process_command(user_input)
            except EOFError:
                # Handle Ctrl+D
                print("\nExiting...")
                self.save_poses()
                running = False
            except KeyboardInterrupt:
                # Handle Ctrl+C
                print("\nExiting...")
                self.save_poses()
                running = False


def main(args=None):
    rclpy.init(args=args)
    
    node = MapAnnotator()
    
    # Spin node in a separate thread
    spin_thread = Thread(target=rclpy.spin, args=(node,), daemon=True)
    spin_thread.start()
    
    # Run the user interface in main thread
    node.run()
    
    # Cleanup
    node.destroy_node()
    rclpy.shutdown()


if __name__ == '__main__':
    main()

```ui markers```

In [ ]:
import rclpy
from rclpy.node import Node
from geometry_msgs.msg import PoseArray
from visualization_msgs.msg import Marker, MarkerArray


class UIMarkers(Node):
    def __init__(self):
        super().__init__('ui_markers')
        
        # Subscribe to PoseArray from map_annotator
        self.pose_array_sub = self.create_subscription(
            PoseArray,
            '/saved_poses',  # Adjust topic name if needed
            self.pose_array_callback,
            10
        )
        
        # Publisher for markers
        self.marker_pub = self.create_publisher(
            MarkerArray,
            '/waypoint_markers',
            10
        )
        
        self.get_logger().info('UI Markers node started')
        
    def pose_array_callback(self, msg):
        """Process incoming PoseArray and create markers"""
        marker_array = MarkerArray()
        
        # Clear previous markers first
        delete_marker = Marker()
        delete_marker.action = Marker.DELETEALL
        marker_array.markers.append(delete_marker)
        
        # Create markers for each pose
        for i, pose in enumerate(msg.poses):
            # Create arrow marker for pose direction
            arrow_marker = self.create_arrow_marker(i, pose, msg.header)
            marker_array.markers.append(arrow_marker)
        
        # Publish all markers
        self.marker_pub.publish(marker_array)
        self.get_logger().info(f'Published {len(msg.poses)} waypoint markers')
    
    def create_arrow_marker(self, idx, pose, header):
        """Create an arrow marker showing pose orientation"""
        marker = Marker()
        marker.header = header
        marker.ns = "arrows"
        marker.id = idx
        marker.type = Marker.ARROW
        marker.action = Marker.ADD
        
        # Set pose
        marker.pose = pose
        
        # Set scale (x=length, y=width, z=height)
        marker.scale.x = 0.5  # Arrow length
        marker.scale.y = 0.1  # Arrow width
        marker.scale.z = 0.1  # Arrow height

        marker.pose.orientation.w = 1.0  # Ensure marker is oriented correctly
        
        # Set color (blue arrows)
        marker.color.r = 0.0
        marker.color.g = 0.5
        marker.color.b = 1.0
        marker.color.a = 0.8
        
        marker.lifetime.sec = 0  # Persist until deleted
        
        return marker


def main(args=None):
    rclpy.init(args=args)
    node = UIMarkers()
    
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        node.destroy_node()
        rclpy.shutdown()


if __name__ == '__main__':
    main()

### Task 2: video

https://drive.google.com/file/d/1Zt0bBXBnD-44YdsvShg6Om3hbrmBILbn/view?usp=sharing

### Task 3: CSV

```json
name,px,py,pz,ox,oy,oz,ow
point4,-2.146065137278835,0.5376242902781916,0.0,0.0,0.0,-0.750462099064031,0.6609134874311527
home,0.00999999999764656,-4.6665582622443615e-15,0.0,0.0,0.0,7.383020323575938e-14,1.0
point5,-2.332014708145742,1.1036704346971247,0.0,0.0,0.0,0.24337549770836672,0.9699321456242208

```